# 04 — Generalisation Test: Cross-Provider (Alibaba 2018)

Tests whether the PPO agent trained **only** on Google Cluster Trace 2011
generalises to a workload derived from the **Alibaba Cluster Trace 2018**,
with **no retraining**. This is a cross-provider generalisation test.

Self-contained: extracts the Alibaba workload parameters from the raw CSV,
then runs the Google-trained agent against the HPA baseline on it.

Uses the refactored modules: `env.py`, `agent.py`, `evaluate.py`.

## 1. Extract Alibaba workload parameters

The Alibaba `machine_usage` trace gives cluster utilisation over time. We use
its CPU-utilisation rhythm to drive the arrival rate (temporal load pattern),
while per-job resource demands are set to small values consistent with the
training distribution (Alibaba's utilisation percentages describe aggregate
cluster load, **not** per-job size — using them as per-job demand was a bug we
corrected).

In [1]:
import pandas as pd
import numpy as np
import json

# ── load Alibaba machine usage (5-min samples over ~8 days) ──────────────
ali = pd.read_csv('alibaba_machine_usage.csv')
print(f"Rows: {len(ali)}  (~{len(ali)*5/60/24:.1f} days)")

ali = ali.iloc[:7*24*12].copy()          # exactly 7 days (12 samples/hr)
ali['interval'] = np.arange(len(ali))
ali['hour_of_day'] = (ali['interval'] // 12) % 24
ali['day_of_week'] = (ali['interval'] // (12*24)) % 7

# convert utilisation rhythm -> arrival rate; per-job demands stay small
UTIL_TO_ARRIVAL = 2500.0

stats_ali = {}
for day in range(7):
    stats_ali[str(day)] = {}
    for hour in range(24):
        bucket = ali[(ali['day_of_week']==day) & (ali['hour_of_day']==hour)]
        if len(bucket) == 0:
            continue
        arrival = float(bucket['cpu_util_percent'].mean() * UTIL_TO_ARRIVAL)
        stats_ali[str(day)][str(hour)] = {
            'arrival_rate': arrival,
            'avg_cpu': 0.04,     # per-JOB demand (small, like Google) — NOT cluster util
            'cpu_std': 0.02,
            'avg_mem': 0.035,    # per-JOB demand — NOT the ~0.87 cluster util
            'mem_std': 0.02,
            'class_distribution': [0.25, 0.25, 0.25, 0.25],
        }

json.dump({'source': 'Alibaba Cluster Trace 2018, machine_usage, 7 days',
           'fidelity': 'utilisation-derived arrival pattern',
           'stats': stats_ali},
          open('alibaba_params.json', 'w'), indent=2)
print("Saved alibaba_params.json\n")
print("Daily arrival pattern (from Alibaba CPU utilisation):")
for day in range(7):
    total = sum(stats_ali[str(day)][str(h)]['arrival_rate'] for h in range(24))
    print(f"  day {day}: {total:>12,.0f}  {'#'*int(total/50000)}")

Rows: 2243  (~7.8 days)
Saved alibaba_params.json

Daily arrival pattern (from Alibaba CPU utilisation):
  day 0:    1,958,053  #######################################
  day 1:    2,379,616  ###############################################
  day 2:    2,489,179  #################################################
  day 3:    2,468,719  #################################################
  day 4:    2,462,433  #################################################
  day 5:    2,281,273  #############################################
  day 6:    2,559,708  ###################################################


## 2. Load the Google-trained agent and run the generalisation test

In [2]:
import torch
from env import CloudClusterEnv, STEPS_PER_WEEK
from agent import ActorCritic
from evaluate import run_ppo, run_hpa

# the Alibaba workload we just extracted
alibaba_stats = json.load(open('alibaba_params.json'))['stats']

# Google-trained model (NO retraining on Alibaba)
gen_net = ActorCritic()
gen_net.load_state_dict(torch.load('ppo_sla-focused.pth'))
gen_net.eval()
print("Loaded Google-trained sla-focused model.\n")

# evaluate PPO and HPA on the Alibaba workload (5-episode average)
def avg_ali(agent_fn, net=None, n=5):
    runs = []
    for i in range(n):
        env = CloudClusterEnv(alibaba_stats, seed=500+i)
        runs.append(agent_fn(env, net) if net is not None else agent_fn(env))
    return {k: float(np.mean([r[k] for r in runs])) for k in ['cost','breaches','util','vms']}

ppo_ali = avg_ali(run_ppo, gen_net)
hpa_ali = avg_ali(run_hpa)

print("="*54)
print("GENERALISATION — Alibaba 2018 (PPO trained ONLY on Google)")
print("="*54)
print(f"{'METRIC':<16}{'HPA':>13}{'PPO':>13}")
print("-"*54)
print(f"{'Total cost':<16}{hpa_ali['cost']:>13.1f}{ppo_ali['cost']:>13.1f}")
print(f"{'SLA breaches':<16}{hpa_ali['breaches']:>13.0f}{ppo_ali['breaches']:>13.0f}")
print(f"{'Utilisation':<16}{hpa_ali['util']:>13.3f}{ppo_ali['util']:>13.3f}")
print(f"{'Avg VMs':<16}{hpa_ali['vms']:>13.1f}{ppo_ali['vms']:>13.1f}")
print("="*54)

cost_d = (ppo_ali['cost']-hpa_ali['cost'])/hpa_ali['cost']*100
# FIX the nan bug: handle HPA having 0 breaches gracefully
if hpa_ali['breaches'] > 0:
    breach_str = f"{(ppo_ali['breaches']-hpa_ali['breaches'])/hpa_ali['breaches']*100:+.1f}%"
else:
    breach_str = "HPA had ~0 breaches on this workload"
print(f"\nPPO vs HPA on Alibaba:  cost {cost_d:+.1f}%,  breaches: {breach_str}")

json.dump({'hpa': hpa_ali, 'ppo': ppo_ali}, open('alibaba_comparison.json','w'), indent=2)
print("Saved alibaba_comparison.json")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Loaded Google-trained sla-focused model.

GENERALISATION — Alibaba 2018 (PPO trained ONLY on Google)
METRIC                    HPA          PPO
------------------------------------------------------
Total cost              573.3        231.6
SLA breaches                0        16112
Utilisation             0.472        0.996
Avg VMs                  17.1          6.9

PPO vs HPA on Alibaba:  cost -59.6%,  breaches: HPA had ~0 breaches on this workload
Saved alibaba_comparison.json


## 3. VM-over-time trace (for the report figure)

Runs the agent through one Alibaba week and records its VM count and breaches
per step, showing how the Google-trained policy behaves on the unseen
workload. Saved for the report figure.

In [3]:
env = CloudClusterEnv(alibaba_stats, seed=500)
obs, _ = env.reset()
obs = torch.tensor(obs, dtype=torch.float32)

vms_over_time, breaches_over_time = [], []
for t in range(STEPS_PER_WEEK):
    with torch.no_grad():
        mean, _ = gen_net.forward(obs.unsqueeze(0))
    obs, r, done, tr, info = env.step(mean.squeeze(0).numpy())
    obs = torch.tensor(obs, dtype=torch.float32)
    vms_over_time.append(info['active_vms'])
    breaches_over_time.append(info['breaches'])
    if done: break

json.dump({'vms': vms_over_time, 'breaches': breaches_over_time},
          open('alibaba_timeseries.json', 'w'))
print(f"Saved alibaba_timeseries.json")
print(f"VM range over Alibaba week: {min(vms_over_time)}-{max(vms_over_time)}, "
      f"mean {np.mean(vms_over_time):.1f}, std {np.std(vms_over_time):.1f}")
print(f"Total breaches over week: {sum(breaches_over_time)}")

Saved alibaba_timeseries.json
VM range over Alibaba week: 2-15, mean 6.8, std 2.7
Total breaches over week: 16948
